In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from crepes import WrapClassifier
from mapie.classification import SplitConformalClassifier
from mapie.calibration import VennAbersCalibrator

from sklearn.calibration import calibration_curve
from sklearn.metrics import log_loss
from sklearn.base import clone
from sklearn.preprocessing import label_binarize

In [ ]:
SEED=1434

In [ ]:
# Cargar el modelo final ya entrenado

modelo = joblib.load('artefactos/pipe_final.joblib')
pipe_final = modelo['modelo']

le = joblib.load('artefactos/label_encoder.joblib')

In [ ]:
# Cargar los conjuntos de datos para las diferentes fases
data = joblib.load('artefactos/datasets_splits.joblib')

x_train = data['x_train']
y_train = data['y_train']

x_calib = data['x_calib']
y_calib = data['y_calib']

x_test  = data['x_test']
y_test  = data['y_test']

In [ ]:
# Intervalo de confianza para los diferentes métodos
ci=0.95

In [ ]:
# Función para calcular las métricas de de los conjuntos de predicción
def evaluar_cp(y_calculados, y_reales, nombres_clases):
    """
    Calcula métricas de predicción conforme:
    - cobertura y cardinalidad global
    - cobertura y cardinalidad por clase
    """

    # Globales
    cobertura = np.mean(y_calculados[np.arange(len(y_reales)), y_reales])
    cardinalidad = y_calculados.sum(axis=1).mean()

    # Por clase
    por_clase = {}
    cardinalidad_por_clase = y_calculados.sum(axis=1)

    
    for cls in range(len(nombres_clases)):
        idx = y_reales == cls
        
        if idx.sum() == 0:
            continue
        
        por_clase[nombres_clases[cls]] = {
            'cobertura': y_calculados[idx, cls].mean(),
            'cardinalidad_media': cardinalidad_por_clase[idx].mean(),
            'n': int(idx.sum())
        }

    return {
        'cobertura_media': cobertura,
        'cardinalidad_media': cardinalidad,
        'por_clase': por_clase
    }

In [ ]:
# Función para mostrar los resultados de métricas calculadas, globales y por clase
def mostrar_resultados_cp(resultados):
    # Global 
    print(f'Cobertura global: {resultados["cobertura_media"]:.4f}')
    print(f'Cardinalidad global: {resultados["cardinalidad_media"]:.4f}')
        
    # Por clase
    for cls, d in resultados['por_clase'].items():
        print(
            f"{cls:15s}: "
            f"cobertura={d['cobertura']:.4f}, "
            f"cardinalidad={d['cardinalidad_media']:.4f} (n={d['n']})"
        )

In [ ]:
# Envolver el clasificador final (no reentrena el modelo, ya está entrenado) - ICP
wrap_crepes = WrapClassifier(pipe_final)

# Ajuste del wrapper conform (no reentrena el modelo, ya está entrenado)
wrap_crepes.fit(x_train, y_train)

# Calibración conforme usando el conjunto de calibración
wrap_crepes.calibrate(x_calib, y_calib)

In [ ]:
# Generación de conjuntos de predicción conformes sobre el conjunto test
# Cada fila de y_sets_crepes contiene el conjunto conforme (codificado como vector binario, con un componente por clase)
y_sets_crepes = wrap_crepes.predict_set(x_test, confidence=ci)

In [ ]:
# Calcular métricas de ICP
resultados_crepes_icp = evaluar_cp(y_sets_crepes, y_test, le.classes_)

In [ ]:
# Mostrar métricas calculadas, globales y por clase
mostrar_resultados_cp(resultados_crepes_icp)

In [ ]:
# Repetir cálculos con Mondrian
# la cobertura se garantiza para cada clase individualmente en lugar de hacerlo de forma global (promedio)
wrap_crepes_mondrian = WrapClassifier(pipe_final)
wrap_crepes_mondrian.fit(x_train, y_train)

wrap_crepes_mondrian.calibrate(x_calib, y_calib, class_cond=True)

y_sets_crepes_mondrian = wrap_crepes_mondrian.predict_set(x_test, confidence=ci)

resultados_crepes_mondrian = evaluar_cp(y_sets_crepes_mondrian, y_test, le.classes_)

mostrar_resultados_cp(resultados_crepes_mondrian)

In [ ]:
# p-values conformes
# Cada fila contiene los p-values conformes de cada clase para cada observación
p_values = wrap_crepes.predict_p(x_test)

p_values_mondrian = wrap_crepes_mondrian.predict_p(x_test)

In [ ]:
# Función para mostrar ejemplos concretos de sujetos de test con los conjuntos calculados
def mostrar_ejemplos_crepes(
    y_sets,
    p_values,
    y_reales,
    nombres_clases,
    ci=0.95,
    n=1,
    clases_objetivo=None
):
    """
    Muestra ejemplos ilustrativos de CREPES.
    
    - Si clases_objetivo es None → muestra los primeros n sujetos del test.
    - Si clases_objetivo es una lista → muestra los primeros n sujetos
      cuya clase real pertenece a esas clases.
    """

    alpha = 1 - ci
    contador = 0

    for i in range(len(y_reales)):
        clase_real = nombres_clases[y_reales[i]]

        # Filtrado por clases si se especifica
        if clases_objetivo is not None and clase_real not in clases_objetivo:
            continue

        print(f'\nSujeto {i}')
        print('  Clase real:', clase_real)
        print('  p-values:')
        for cls, p in zip(nombres_clases, p_values[i]):
            print(f"    {cls:15s}: p = {p:.3f}")
        print(
            '  Conjunto conforme:',
            [cls for cls, p in zip(nombres_clases, p_values[i]) if p >= alpha]
        )

        contador += 1
        if contador == n:
            break

In [ ]:
# Mostrar 5 ejemplos entre todas las clases
mostrar_ejemplos_crepes(
    y_sets_crepes,
    p_values,
    y_test,
    le.classes_,
    ci=ci,
    n=5
)

In [ ]:
# Mostrar 5 ejemplos entre todas las clases (Mondrian)
mostrar_ejemplos_crepes(
    y_sets_crepes_mondrian,
    p_values_mondrian,
    y_test,
    le.classes_,
    ci=ci,
    n=5
)

In [ ]:
# Mostrar 5 ejemplos entre las clases minoritarias
mostrar_ejemplos_crepes(
    y_sets_crepes,
    p_values,
    y_test,
    le.classes_,
    ci=ci,
    n=5,
    clases_objetivo=['Type 1', 'Gestational']
)

In [ ]:
# Mostrar 5 ejemplos entre las clases minoritarias (Mondrian)
mostrar_ejemplos_crepes(
    y_sets_crepes_mondrian,
    p_values_mondrian,
    y_test,
    le.classes_,
    ci=ci,
    n=5,
    clases_objetivo=['Type 1', 'Gestational']
)

In [ ]:
# Función para mostrar ejemplos concretos de sujetos de test con los conjuntos APS y RAPS
def mostrar_ejemplos_mapie(
    y_sets,
    y_reales,
    nombres_clases,
    n=1,
    clases_objetivo=None
):
    """
    Muestra ejemplos ilustrativos de APS y RAPS.
    - y_sets contiene directamente los conjuntos conformes en formato binario (Verdadero/Falso)
    - Si clases_objetivo es None → muestra los primeros n sujetos del test.
    - Si clases_objetivo es una lista → muestra los primeros n sujetos
      cuya clase real pertenece a esas clases.
    """

    contador = 0

    for i in range(len(y_reales)):
        clase_real = nombres_clases[y_reales[i]]

        # Filtrado por clases si se especifica
        if clases_objetivo is not None and clase_real not in clases_objetivo:
            continue

        print(f'\nSujeto {i}')
        print('  Clase real:', clase_real)
        print(
            '  Conjunto conforme:',
            [cls for cls, included in zip(nombres_clases, y_sets[i]) if included]
        )

        contador += 1
        if contador == n:
            break

In [ ]:
# Inicialización de APS con el clasificador final
# Se usa prefit=True para indicar que el modelo ya está entrenado y no debe reentrenarse
mapie_aps = SplitConformalClassifier(
    estimator=pipe_final,
    confidence_level=ci,
    conformity_score='aps',
    prefit=True
)

# Calibración conforme utilizando el conjunto de calibración
mapie_aps.conformalize(x_calib, y_calib)

In [ ]:
# Generación de conjuntos conformes sobre el conjunto test
# y_pred_point_aps: predicción puntual (clasificación clásica)
# y_sets_mapie_aps: conjunto conforme (posibles clases con nivel de confianza dado)
y_pred_point_aps, y_sets_mapie_aps = mapie_aps.predict_set(x_test)

In [ ]:
resultados_mapie_aps = evaluar_cp(y_sets_mapie_aps, y_test, le.classes_)

In [ ]:
mostrar_resultados_cp(resultados_mapie_aps)

In [ ]:
# Mostrar 5 ejemplos entre todas las clases
mostrar_ejemplos_mapie(
    y_sets_mapie_aps,
    y_test,
    le.classes_,
    n=5
)

In [ ]:
# Mostrar 5 ejemplos entre las clases minoritarias
mostrar_ejemplos_mapie(
    y_sets_mapie_aps,
    y_test,
    le.classes_,
    n=5,
    clases_objetivo=['Type 1', 'Gestational']
)

In [ ]:
# Inicialización de RAPS con el clasificador final
# Se usa prefit=True para indicar que el modelo ya está entrenado y no debe reentrenarse
mapie_raps = SplitConformalClassifier(
    estimator=pipe_final,
    confidence_level=ci,
    conformity_score='raps',
    prefit=True
)

# Calibración conforme utilizando el conjunto de calibración
mapie_raps.conformalize(x_calib, y_calib)

# Generación de conjuntos conformes sobre el conjunto test
y_pred_point_raps, y_sets_mapie_raps = mapie_raps.predict_set(x_test)

resultados_mapie_raps = evaluar_cp(y_sets_mapie_raps, y_test, le.classes_)

In [ ]:
mostrar_resultados_cp (resultados_mapie_raps)

In [ ]:
# Mostrar 5 ejemplos entre todas las clases
mostrar_ejemplos_mapie(
    y_sets_mapie_raps,
    y_test,
    le.classes_,
    n=5
)

In [ ]:
# Mostrar 5 ejemplos entre las clases minoritarias
mostrar_ejemplos_mapie(
    y_sets_mapie_raps,
    y_test,
    le.classes_,
    n=5,
    clases_objetivo=['Type 1', 'Gestational']
)

In [ ]:
# Inicialización del calibrador Venn‑Abers
va_mapie = VennAbersCalibrator(
    estimator=clone(pipe_final),
    inductive=True,
    random_state=SEED
)

# Calibración probabilística utilizando el conjunto de calibración
va_mapie.fit(
    X=x_train,
    y=y_train,
    X_calib=x_calib,
    y_calib=y_calib
)

# Predicción de probabilidades calibradas sobre el conjunto test
proba_calibradas = va_mapie.predict_proba(x_test)

# Probabilidades sin calibrar
proba_sin_calibrar = pipe_final.predict_proba(x_test)


print('¿NaN en calibradas?:', np.isnan(proba_calibradas).any())

# Suma de probabilidades por fila (debe ser 1)
print('Suma media de probabilidades calibradas:', proba_calibradas.sum(axis=1).mean())
print('Suma min:', proba_calibradas.sum(axis=1).min())
print('Suma max:', proba_calibradas.sum(axis=1).max())


logloss_sin_calibrar = log_loss(y_test, proba_sin_calibrar)
logloss_calibradas  = log_loss(y_test, proba_calibradas)

print('LogLoss sin calibrar:', logloss_sin_calibrar)
print('LogLoss Venn‑Abers:', logloss_calibradas)

In [ ]:
def brier_multiclase(y_true, proba, n_classes):
    Y = label_binarize(y_true, classes=np.arange(n_classes))
    return np.mean(np.sum((proba - Y) ** 2, axis=1))

n_classes = len(le.classes_)

brier_sin_calibrar = brier_multiclase(y_test, proba_sin_calibrar, n_classes)
brier_calibradas  = brier_multiclase(y_test, proba_calibradas, n_classes)

print('Brier sin calibrar:', brier_sin_calibrar)
print('Brier Venn‑Abers:', brier_calibradas)

In [ ]:
resultados_toplabel = {}
# Probabilidad de la clase predicha
conf_sin_calibrar = np.max(proba_sin_calibrar, axis=1)
conf_calibradas = np.max(proba_calibradas, axis=1)

y_pred_sin = np.argmax(proba_sin_calibrar, axis=1)
y_pred_cal = np.argmax(proba_calibradas, axis=1)

# Aciertos para le modelo calibrado y sin calibrar
correct_sin = (y_pred_sin == y_test).astype(int)
correct_cal = (y_pred_cal == y_test).astype(int)

# Curvas con estrategia quantile para que incluya el mismo número de muestras en cada intervalo considerado (10 intervalos)
frac_pos_sin_calibrar, mean_pred_sin_calibrar = calibration_curve(correct_sin, conf_sin_calibrar, n_bins=10, strategy='quantile')
frac_pos_calibradas, mean_pred_calibradas = calibration_curve(correct_cal, conf_calibradas, n_bins=10, strategy='quantile')


resultados_toplabel = {
    "sin_calibrar": {
        "mean_pred": mean_pred_sin_calibrar,
        "frac_pos": frac_pos_sin_calibrar
    },
    "calibrado": {
        "mean_pred": mean_pred_calibradas,
        "frac_pos": frac_pos_calibradas
    }
}

plt.figure(figsize=(7, 6))

# Línea de calibración perfecta
plt.plot([0,1], [0,1], '--', linewidth=1, label='Perfectamente calibrado', color='gray')

# Curvas para el modelo calibrado y sin calibrar
plt.plot(mean_pred_sin_calibrar, frac_pos_sin_calibrar, 'o-', linewidth=2, markersize=4, label='Sin calibrar', color='C0')
plt.plot(mean_pred_calibradas, frac_pos_calibradas, 'o-', linewidth=2, markersize=4, label='Calibrado (Venn‑Abers)', color='C1')

plt.xlabel('Probabilidad predicha (top‑label)')
plt.ylabel('Frecuencia observada')
plt.title('Diagrama de fiabilidad (top‑label, conjunto de prueba)')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
clases = le.classes_

resultados_one_rest = {}

fig, axes = plt.subplots(3, 2, figsize=(15, 15))
axes = axes.ravel()

for i, nombre_clase in enumerate(clases):
    ax = axes[i]
    
    # Enfoque One-vs-Rest
    y_true_clase = (y_test == i).astype(int)

    # Las probabilidades asignadas específicamente a esta clase
    proba_sin_clase = proba_sin_calibrar[:, i]
    proba_cal_clase = proba_calibradas[:, i]
    
    # Curvas con estrategia quantile para que incluya el mismo número de muestras en cada intervalo considerado (10 intervalos)
    frac_pos_sin, mean_pred_sin = calibration_curve(y_true_clase, proba_sin_clase, n_bins=10, strategy='quantile')
    frac_pos_cal, mean_pred_cal = calibration_curve(y_true_clase, proba_cal_clase, n_bins=10, strategy='quantile')

    # Curvas calibradas y calibración perfecta para cada clase
    ax.plot([0, 1], [0, 1], '--', color='gray', linewidth=1, label='Perfectamente calibrado')
    ax.plot(mean_pred_sin, frac_pos_sin, 'o-', linewidth=2, markersize=4, label='Sin calibrar', color='C0')
    ax.plot(mean_pred_cal, frac_pos_cal, 'o-', linewidth=2, markersize=4, label='Calibrado (Venn‑Abers)', color='C1')

    resultados_one_rest[nombre_clase] = {
        "sin_calibrar": {
            "mean_pred": mean_pred_sin,
            "frac_pos": frac_pos_sin
        },
        "calibrado": {
            "mean_pred": mean_pred_cal,
            "frac_pos": frac_pos_cal
        }
    }
    
    # Formato de cada subgráfico
    ax.set_xlabel('Probabilidad predicha')
    ax.set_ylabel('Frecuencia observada')
    ax.set_title(f'Clase: {nombre_clase}')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.0])
    ax.grid(alpha=0.3)

# Leyenda en 6 gráfico
ax_leyenda = axes[5]
ax_leyenda.axis('off')

ax_leyenda.plot([], [], '--', color='gray', linewidth=1, label='Perfectamente calibrado')
ax_leyenda.plot([], [], 'o-', linewidth=2, markersize=4, label='Sin calibrar', color='C0')
ax_leyenda.plot([], [], 'o-', linewidth=2, markersize=4, label='Calibrado (Venn‑Abers)', color='C1')

ax_leyenda.legend(loc='center', fontsize=20, frameon=True, shadow=True, borderpad=1)

plt.suptitle('Diagramas de fiabilidad desagregados clase (Conjunto de prueba)')
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar los valores de los diagramas de fiabilidad
df_toplabel = pd.DataFrame({
    "Pred (sin cal)": resultados_toplabel["sin_calibrar"]["mean_pred"],
    "Real (sin cal)": resultados_toplabel["sin_calibrar"]["frac_pos"],
    "Pred (cal)": resultados_toplabel["calibrado"]["mean_pred"],
    "Real (cal)": resultados_toplabel["calibrado"]["frac_pos"],
}).round(4)

print(df_toplabel)

In [ ]:
# Mostrar los valores de los diagramas de fiabilidad
tablas_por_clase = {}

for clase, datos in resultados_one_rest.items():
    
    df_clase = pd.DataFrame({
        "Pred (sin cal)": datos["sin_calibrar"]["mean_pred"],
        "Real (sin cal)": datos["sin_calibrar"]["frac_pos"],
        "Pred (cal)": datos["calibrado"]["mean_pred"],
        "Real (cal)": datos["calibrado"]["frac_pos"],
    })
    
    tablas_por_clase[clase] = df_clase.round(4)

print(tablas_por_clase)

In [ ]:
def mostrar_resultados_calibracion(logloss_raw, logloss_cal, brier_raw, brier_cal):
    print(f'LogLoss sin calibrar: {logloss_raw:.4f}')
    print(f'LogLoss Venn‑Abers: {logloss_cal:.4f}')
    print(f'Brier sin calibrar: {brier_raw:.4f}')
    print(f'Brier Venn‑Abers: {brier_cal:.4f}')

In [ ]:
mostrar_resultados_calibracion(
    logloss_sin_calibrar,
    logloss_calibradas,
    brier_sin_calibrar,
    brier_calibradas
)

In [ ]:
# Función para calcular logLoss por clase
def logloss_por_clase(y_true, proba, class_names):
    res = {}
    for cls in range(len(class_names)):
        idx = (y_true == cls)
        if idx.sum() == 0:
            continue

        p_cls = proba[idx, cls]
        y_bin = np.ones_like(p_cls)

        res[class_names[cls]] = round(float(log_loss(y_bin, p_cls, labels=[0, 1])), 4)

    return res

In [ ]:
ll_sin_calibrar_por_class = logloss_por_clase(y_test, proba_sin_calibrar, le.classes_)
ll_calibradas_por_class  = logloss_por_clase(y_test, proba_calibradas, le.classes_)

print('LogLoss por clase (sin calibrar):', ll_sin_calibrar_por_class)
print('LogLoss por clase (Venn‑Abers):', ll_calibradas_por_class)

In [ ]:
# Función para calcular Brier score por clase
def brier_por_clase(y_true, proba, class_names):
    res = {}
    for cls in range(len(class_names)):
        idx = (y_true == cls)
        if idx.sum() == 0:
            continue

        p_cls = proba[idx, cls]
        y_bin = np.ones_like(p_cls)

        res[class_names[cls]] = round(float(np.mean((p_cls - y_bin) ** 2)), 4)

    return res

In [ ]:
brier_sin_calibrar_por_class = brier_por_clase(y_test, proba_sin_calibrar, le.classes_)
brier_calibradas_por_class  = brier_por_clase(y_test, proba_calibradas, le.classes_)

print('Brier por clase (sin calibrar):', brier_sin_calibrar_por_class)
print('Brier por clase (Venn‑Abers):', brier_calibradas_por_class)

In [ ]:
# Guardar los resultados
joblib.dump(
    {
        'logloss': {
            'sin_calibrar': logloss_sin_calibrar,
            'venn_abers': logloss_calibradas
        },
        'brier': {
            'sin_calibrar': brier_sin_calibrar,
            'venn_abers': brier_calibradas
        },
        'logloss_por_clase': {
            'sin_calibrar': ll_sin_calibrar_por_class,
            'venn_abers': ll_calibradas_por_class
        },
        'brier_por_clase': {
            'sin_calibrar': brier_sin_calibrar_por_class,
            'venn_abers': brier_calibradas_por_class
        },
        'conjuntos':
        {
            'icp': resultados_crepes_icp,
            'mondrian': resultados_crepes_mondrian,
            'mapie_aps': resultados_mapie_aps,
            'mapie_raps': resultados_mapie_raps
        }
    },
    'artefactos/resultados_finales_cp.joblib'
)

joblib.dump(
    {
        "one_vs_rest": tablas_por_clase,
        "top_label": df_toplabel
    },
    "artefactos/diagramas_fiabilidad.joblib"
)